# Cleaned Odyssey's text from Robert Fitzgerald's translation

### Before

- Scanned from PDF. Here are some characteristics. 
- Noisy but
    * Line numbering every paragraph
    * Subtitle scheme:  
        `Book Two`  
        `A HERO'S SON AWAKENS`  
        `LINES 1-22`
    * Pagination
    * Page-headers
    * Intro and ToC
    * Glossary and notes at the end
    * Separated by books

### After
- Only the text 
- Keep book numbers: 'Book I',…'Book XXIV'
- No numeral digits (if in book title, changed to Roman)

In [1]:
translator = "Fitzgerald"
filepath = f"/Users/debr/odysseys_en/raw_txts/Odyssey_{translator}.txt"

# Define start and end markers
start_marker = "Book One\n"
end_marker = "though still she kept the form and voice of Mentor."

In [2]:
def extract_text_between_markers(file_path, start_marker, end_marker, skip_empty_lines=True):
    """
    Extract text between start and end markers from a text file.
    
    Args:
        file_path (str): Path to the text file
        start_marker (str): Text that marks the beginning of the section to extract
        end_marker (str): Text that marks the end of the section to extract
        skip_empty_lines (bool): Whether to skip empty lines in the output
    
    Returns:
        list: List of strings, each representing a line in the extracted text
    """
    # Read the entire file content
    with open(file_path, "r", encoding="utf-8") as inputfile:
        file_content = inputfile.read()
    
    # Find the start and end positions
    start_pos = file_content.find(start_marker)
    end_pos = file_content.find(end_marker)
    
    # Handle cases where markers aren't found
    if start_pos == -1:
        print(f"Warning: Start marker '{start_marker}' not found in the file.")
        start_pos = 0
    else:
        # Include the start marker in the output
        start_pos = start_pos
    
    if end_pos == -1:
        print(f"Warning: End marker '{end_marker}' not found in the file.")
        end_pos = len(file_content)
    else:
        # Include the end marker in the output
        end_pos = end_pos + len(end_marker)
    
    # Extract the text between markers
    extracted_text = file_content[start_pos:end_pos]
    
    # Split the extracted text into lines
    lines = extracted_text.splitlines()
    
    # Filter out empty lines if requested
    if skip_empty_lines:
        lines = [line for line in lines if line.strip()]
    
    return lines



# Extract the text
extracted_lines = extract_text_between_markers(filepath, start_marker, end_marker)

# Verify by printing the end of the extracted text
print(f"Tell me, Python, how {translator}'s Odyssey starts:")
print("\n".join(extracted_lines[:4]))

print(f"\nO, but tell me, Python, how {translator}'s Odyssey ends:")
print("\n".join(extracted_lines[-3:]))

Tell me, Python, how Fitzgerald's Odyssey starts:
Book One
A GODDESS INTERVENES
LINES 1-15
Sing in me, Muse, and through me tell the story

O, but tell me, Python, how Fitzgerald's Odyssey ends:
set by their arbiter, Athena, daughter
of Zeus who bears the stormcloud as a shield-
though still she kept the form and voice of Mentor.


In [3]:
# Step 3: Find subtitle-couplets after "Book…" and remove them
import re
def process_books(lines):
    cleaned_books = []
    current_book = []
    
    for line in lines:
        line = line.strip()  # Remove leading/trailing spaces

        # Skip completely empty lines
        if not line:
            continue  

        # Identify the start of a new book
        if line.startswith("Book "):
            if current_book:  # Process the previous book if it exists
                cleaned_books.extend(process_book_section(current_book))
            current_book = [line]  # Start a new book entry
        else:
            current_book.append(line)
    
    # Process the last book
    if current_book:
        cleaned_books.extend(process_book_section(current_book))

    return cleaned_books

def process_book_section(book_lines):
    """
    Extract the first 4 lines of a book while:
    - Removing all empty lines after "Book …"
    - Pruning lines 2 & 3
    - Keeping all remaining content
    """
    result = []
    result.append(book_lines[0])  # Keep the book title

    # Remove empty lines after "Book …"
    filtered_lines = [line for line in book_lines[1:] if line.strip()]

    # Ensure at least 4 lines exist before pruning
    if len(filtered_lines) > 0:
        result.append("")  # Prune line 2
    if len(filtered_lines) > 1:
        result.append("")  # Prune line 3
    if len(filtered_lines) > 2:
        result.extend(filtered_lines[2:])  # Keep the rest of the book

    return result

skimmed_lines = process_books(extracted_lines)

# Checking and failing to prune empty lines
print("\n".join(skimmed_lines[:5]))

Book One


Sing in me, Muse, and through me tell the story
of that man skilled in all ways of contending,


In [5]:
def remove_lines_with_patterns(lines_list, patterns_to_remove):
    """
    Remove lines that contain any of the specified patterns anywhere in the line.
    
    Args:
        lines_list (list): List of strings to filter
        patterns_to_remove (list): List of string patterns to search for in each line
    
    Returns:
        list: Filtered list with matching lines removed
    """
    filtered_lines = []
    
    for line in lines_list:
        should_keep = True
        
        # Check if the line contains any of the patterns
        for pattern in patterns_to_remove:
            if pattern in line:  # Simple string containment check (case sensitive)
                should_keep = False
                break
        
        if should_keep:
            filtered_lines.append(line)
            
    return filtered_lines

# Example usage
extracted_lines = [...]  # Your existing list of strings

# Define patterns to remove (case sensitive)
patterns = ["BOOK", "LI"]

# Filter the lines
filtered_lines = remove_lines_with_patterns(skimmed_lines, patterns)

# Verify the filtering
removed_count = len(skimmed_lines) - len(filtered_lines)
print(f"Removed {removed_count} lines containing the patterns.")

Removed 368 lines containing the patterns.


In [7]:
# Because those persistent empty lines, brute force to remove them
filtered_lines  = [line for line in filtered_lines if line.strip()]
print(filtered_lines)

['Book One', 'Sing in me, Muse, and through me tell the story', 'of that man skilled in all ways of contending,', 'the wanderer, harried for years on end,', 'after he plundered the stronghold', 'on the proud height of Troy.', 'He saw the townlands', 'and learned the minds of many distant men,', 'and weathered many bitter nights and days', 'in his deep heart at sea, while he fought only', 'to save his life, to bring his shipmates home.', 'But not by will nor valor could he save them,', 'for their own recklessness destroyed them all-', 'children and fools, they killed and feasted on', 'the cattle of Lord Hehos, the Sun,', 'and he who moves all day through heaven', 'took from their eyes the dawTi of their return.', 'Of these adventures. Muse, daughter of Zeus,', 'tell us in our time, lift the great song again.', 'Begin when all the rest who left behind them', 'headlong death in battle or at sea', 'had long ago returned, while he alone still hungered', 'for home and wife. Her ladyship Kaly

In [ ]:
patterns = {'\\v': 'w', 
            \\^dll
            '\\-': 'y'
            '\\\\-i\\': 
            }

In [8]:
# Extrated lines to text
final_text = "\n".join(filtered_lines)

# Write the cleaned content to a new file for further processing
output_filepath = f"/Users/debr/odysseys_en/cleaned_txts/Odyssey_{translator}_cleaned.txt"
with open(output_filepath, "w", encoding="utf-8") as outputfile:
    outputfile.writelines(final_text)